# EEG_08b — Ablation Study: Metodi di Costruzione del Grafo

Questo notebook analizza i risultati dell'esperimento EEG_08, confrontando **5 metodi di costruzione del grafo** (PCC, PLV, wPLI, Learned, Dynamic) e **3-4 architetture ChebGCN** su un task di classificazione a 4 categorie semantiche (chance level = 25%).

**Obiettivo**: capire se il metodo di costruzione del grafo influenza le performance, o se tutti i modelli collassano sulla classe maggioritaria (class collapse), e identificare qual è il trade-off velocità/accuratezza.

**Env**: `daniele_311` | **Dataset**: EEG imagined speech, 70 soggetti, 4 classi semantiche

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

# ---------------------------------------------------------------------------
# Risultati hardcoded (i CSV sono sulla VM remota)
# (method, model, val_acc, val_bacc, test_acc, test_bacc, time_s)
# ---------------------------------------------------------------------------
results = [
    ("PCC",     "ChebGCN_2L",      0.2035, 0.2527, 0.2018, 0.2490, 142.2),
    ("PCC",     "ChebGCN_3L",      0.2387, 0.2607, 0.2251, 0.2445, 303.6),
    ("PCC",     "ChebGCN_Skip",    0.3942, 0.2538, 0.3875, 0.2524, 189.8),
    ("PLV",     "ChebGCN_2L",      0.3833, 0.2557, 0.3555, 0.2437, 205.8),
    ("PLV",     "ChebGCN_3L",      0.3178, 0.2522, 0.3222, 0.2469, 197.7),
    ("PLV",     "ChebGCN_Skip",    0.4002, 0.2503, 0.3999, 0.2499, 225.5),
    ("wPLI",    "ChebGCN_2L",      0.1733, 0.2511, 0.1755, 0.2514, 171.0),
    ("wPLI",    "ChebGCN_3L",      0.3913, 0.2528, 0.3915, 0.2512, 224.1),
    ("wPLI",    "ChebGCN_Skip",    0.3482, 0.2535, 0.3463, 0.2515, 163.6),
    ("Learned", "ChebGCN_Learned", 0.1829, 0.2528, 0.1753, 0.2430, 6219.5),
    ("Dynamic", "ChebGCN_Dynamic", 0.2464, 0.2501, 0.2481, 0.2485, 7410.6),
]

df = pd.DataFrame(results, columns=["method", "model", "val_acc", "val_bacc", "test_acc", "test_bacc", "time_s"])
df["time_min"] = df["time_s"] / 60.0

CHANCE = 0.25

# Trova la root del progetto tramite .git
_here = Path(".").resolve()
project_root = _here
for p in [_here, *_here.parents]:
    if (p / ".git").exists():
        project_root = p
        break
figures_dir = project_root / "figures"
figures_dir.mkdir(exist_ok=True)

print(f"Project root: {project_root}")
print(f"Figures dir:  {figures_dir}")
print()
print(df.to_string(index=False))

## Plot 1 — Ablation: val_bacc per Metodo × Architettura

Il grafico mostra la **balanced accuracy** (val_bacc) per ogni combinazione metodo-architettura.  
La balanced accuracy è la metrica corretta in presenza di class imbalance: un valore pari a 0.25 indica che il modello non apprende nulla (chance level su 4 classi).

In [ ]:
from IPython.display import Image, display

plot1_path = figures_dir / "eeg08_ablation_graph_methods.png"

if plot1_path.exists():
    display(Image(filename=str(plot1_path)))
    print(f"Figura caricata da: {plot1_path}")
else:
    print(f"[INFO] File non trovato localmente: {plot1_path}")
    print("Questa figura è stata generata sulla VM remota durante il training EEG_08.")
    print("Eseguire il notebook EEG_08 sulla VM per rigenerarla.")

## Plot 2 — val_acc vs val_bacc: Class Collapse

Scatter plot che mette in evidenza il fenomeno di **class collapse**: alcuni modelli raggiungono una `val_acc` elevata (fino a ~40%) ma la loro `val_bacc` rimane a ~25%, il che indica che predicono quasi sempre la classe maggioritaria anziché aver imparato a distinguere le categorie.

Una `val_bacc` > 0.25 è il vero indicatore di apprendimento.

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

METHOD_COLORS = {
    "PCC":     "#4C72B0",
    "PLV":     "#DD8452",
    "wPLI":    "#55A868",
    "Learned": "#C44E52",
    "Dynamic": "#8172B2",
}

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)

for _, row in df.iterrows():
    color = METHOD_COLORS.get(row["method"], "gray")
    ax.scatter(row["val_acc"], row["val_bacc"], color=color, s=100, zorder=5,
               edgecolors="white", linewidths=0.6)
    # Annotazione con nome modello (font piccolo)
    short_name = row["model"].replace("ChebGCN_", "")
    ax.annotate(
        short_name,
        xy=(row["val_acc"], row["val_bacc"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=7.5,
        color="black",
        alpha=0.85,
    )

# Linee di chance level
ax.axvline(CHANCE, color="gray", linestyle="--", linewidth=1.2, alpha=0.7, label="Chance (val_acc = 0.25)")
ax.axhline(CHANCE, color="tomato", linestyle="--", linewidth=1.2, alpha=0.7, label="Chance (val_bacc = 0.25)")

# Legenda metodi
legend_patches = [
    mpatches.Patch(color=c, label=m) for m, c in METHOD_COLORS.items()
]
legend1 = ax.legend(handles=legend_patches, title="Metodo Grafo", loc="upper left",
                    fontsize=9, title_fontsize=9)
ax.add_artist(legend1)
ax.legend(loc="lower right", fontsize=8)

ax.set_xlabel("val_acc (accuratezza semplice)", fontsize=11)
ax.set_ylabel("val_bacc (balanced accuracy)", fontsize=11)
ax.set_title(
    "val_acc vs val_bacc — modelli con alta val_acc ma bacc=25%\ncollassano su classe maggioritaria",
    fontsize=11, pad=12
)

# Annotazione zona collapse
ax.annotate(
    "Zona class collapse\n(val_acc alto, bacc ≈ chance)",
    xy=(0.38, 0.2503), xytext=(0.28, 0.247),
    fontsize=8, color="tomato",
    arrowprops=dict(arrowstyle="->", color="tomato", lw=1.2),
)

ax.set_xlim(0.13, 0.45)
ax.set_ylim(0.248, 0.264)

plt.tight_layout()
out_path = figures_dir / "eeg08_acc_vs_bacc.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato in: {out_path}")

## Plot 3 — Heatmap: val_bacc per Metodo × Architettura

La heatmap mostra come la balanced accuracy vari al variare del metodo di costruzione del grafo (righe) e dell'architettura GCN (colonne). Le combinazioni inesistenti sono lasciate bianche (NaN).

Colore verde = val_bacc > 0.25 (sopra chance), rosso = sotto chance.

In [ ]:
# Mappa il nome del modello alla colonna corta per la heatmap
arch_map = {
    "ChebGCN_2L":      "2L",
    "ChebGCN_3L":      "3L",
    "ChebGCN_Skip":    "Skip",
    "ChebGCN_Learned": "Learned",
    "ChebGCN_Dynamic": "Dynamic",
}
df["arch"] = df["model"].map(arch_map)

METHODS = ["PCC", "PLV", "wPLI", "Learned", "Dynamic"]
ARCHS   = ["2L", "3L", "Skip", "Learned", "Dynamic"]

# Costruisci la matrice (NaN dove la combinazione non esiste)
heatmap_data = pd.DataFrame(np.nan, index=METHODS, columns=ARCHS)
for _, row in df.iterrows():
    heatmap_data.loc[row["method"], row["arch"]] = row["val_bacc"]

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
sns.heatmap(
    heatmap_data,
    ax=ax,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=CHANCE,
    vmin=0.245,
    vmax=0.265,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "val_bacc", "shrink": 0.8},
    annot_kws={"size": 10},
)
ax.set_title("Heatmap val_bacc — Metodo Grafo × Architettura\n(verde = sopra chance 0.25, rosso = sotto)",
             fontsize=11, pad=12)
ax.set_xlabel("Architettura GCN", fontsize=11)
ax.set_ylabel("Metodo Costruzione Grafo", fontsize=11)
ax.tick_params(axis="x", labelsize=10)
ax.tick_params(axis="y", labelsize=10, rotation=0)

plt.tight_layout()
out_path = figures_dir / "eeg08_heatmap_bacc.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato in: {out_path}")

## Plot 4 — Tempo di Training per Metodo

I metodi **Learned** (~103 min) e **Dynamic** (~123 min) sono circa **50× più lenti** dei metodi basati su connettività funzionale (PCC/PLV/wPLI: 2-5 min), senza alcun guadagno in balanced accuracy.

Questo rende i grafi pre-calcolati (PCC/PLV/wPLI) la scelta più efficiente per il prossimo step.

In [ ]:
df_time = df.sort_values("time_min", ascending=True).copy()
labels = [f"{row['method']} / {row['arch']}" for _, row in df_time.iterrows()]
times  = df_time["time_min"].values
colors = [METHOD_COLORS.get(m, "gray") for m in df_time["method"]]

fig, ax = plt.subplots(figsize=(10, 5), dpi=150)
bars = ax.barh(labels, times, color=colors, edgecolor="white", height=0.65)

# Annotazioni sul valore
for bar, t in zip(bars, times):
    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{t:.1f} min",
        va="center", ha="left", fontsize=8.5,
    )

# Linea di riferimento: media dei metodi veloci
fast_methods = df_time[df_time["method"].isin(["PCC", "PLV", "wPLI"])]
avg_fast = fast_methods["time_min"].mean()
ax.axvline(avg_fast, color="steelblue", linestyle=":", linewidth=1.5,
           label=f"Media PCC/PLV/wPLI ({avg_fast:.1f} min)")
ax.axvline(100, color="tomato", linestyle=":", linewidth=1.5,
           label="100 min soglia")

# Legenda metodi
legend_patches = [mpatches.Patch(color=c, label=m) for m, c in METHOD_COLORS.items()]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][-2:],
          fontsize=8, loc="lower right")

ax.set_xlabel("Tempo di training (minuti)", fontsize=11)
ax.set_title(
    "Tempo di Training — Learned e Dynamic ~50× più lenti, stessa accuracy\n"
    "(val_bacc ≈ 0.25 per tutti i metodi)",
    fontsize=11, pad=12
)
ax.set_xlim(0, df_time["time_min"].max() * 1.15)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
out_path = figures_dir / "eeg08_training_time.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Salvato in: {out_path}")

## Plot 5 — Struttura Grafi: Stesso Trial × 5 Metodi

Visualizzazione comparativa dei **5 grafi di connettività** costruiti dallo stesso trial EEG (soggetto 0, epoca 0). Ogni nodo rappresenta un elettrodo, colorato per regione cerebrale approssimativa. Lo spessore degli archi è proporzionale al peso della connessione.

> **Nota**: questa cella necessita dei dataset pre-calcolati (`data/interim/graphs/`) disponibili sulla VM remota. In esecuzione locale viene saltata automaticamente.

In [ ]:
import math

def read_eloc(path):
    """Legge posizioni elettrodi dal file .locs (formato EEGLAB)."""
    pos = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                try:
                    theta  = float(parts[1])
                    radius = float(parts[2])
                    name   = parts[3]
                    th = math.radians(theta)
                    pos[name] = (radius * math.sin(th), radius * math.cos(th))
                except Exception:
                    pass
    return pos


def get_region_color(name):
    """Colore per regione cerebrale basato sul prefisso del nome elettrodo."""
    n = name.upper()
    if n.startswith("F"):
        return "#4C72B0"   # frontale — blu
    elif n.startswith("T"):
        return "#55A868"   # temporale — verde
    elif n.startswith("P"):
        return "#DD8452"   # parietale — arancio
    elif n.startswith("O"):
        return "#C44E52"   # occipitale — rosso
    elif n.startswith("C"):
        return "#8172B2"   # centrale — viola
    else:
        return "#999999"   # altro — grigio


try:
    import torch
    from torch_geometric.data import Data

    locs_path = project_root / "src" / "io" / "ebneuro.locs"
    graphs_dir = project_root / "data" / "interim" / "graphs"

    if not locs_path.exists():
        raise FileNotFoundError(f"File .locs non trovato: {locs_path}")
    if not graphs_dir.exists():
        raise FileNotFoundError(f"Cartella grafi non trovata: {graphs_dir}")

    # Leggi posizioni elettrodi
    eloc = read_eloc(locs_path)

    # Dataset da caricare (nome file → etichetta)
    datasets = {
        "PCC":     "dataset_pcc_k6.pt",
        "PLV":     "dataset_plv_k6.pt",
        "wPLI":    "dataset_wpli_k6.pt",
        "Learned": "dataset_learned_k6.pt",
        "Dynamic": "dataset_dynamic_k6.pt",
    }

    fig, axes = plt.subplots(1, 5, figsize=(20, 4.5), dpi=150)
    fig.suptitle(
        "Struttura del Grafo per lo Stesso Trial (soggetto 0, epoca 0) × 5 Metodi",
        fontsize=12, y=1.02
    )

    for ax, (method, fname) in zip(axes, datasets.items()):
        fpath = graphs_dir / fname
        ax.set_title(method, fontsize=11, fontweight="bold")
        ax.set_aspect("equal")
        ax.axis("off")

        if not fpath.exists():
            ax.text(0.5, 0.5, f"File non trovato:\n{fname}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=8, color="gray")
            continue

        data_list = torch.load(fpath, map_location="cpu", weights_only=False)
        sample = data_list[0] if isinstance(data_list, list) else data_list

        # Nomi elettrodi nell'ordine del tensore (presi dal .locs, esclusi A1/A2)
        all_names  = list(eloc.keys())
        valid_names = [n for n in all_names if n not in ("A1", "A2")]
        n_nodes    = sample.edge_index.max().item() + 1
        node_names = valid_names[:n_nodes]

        xs = np.array([eloc.get(n, (0, 0))[0] for n in node_names])
        ys = np.array([eloc.get(n, (0, 0))[1] for n in node_names])
        node_colors = [get_region_color(n) for n in node_names]

        # Disegna archi
        ei = sample.edge_index.numpy()
        edge_weight = None
        if hasattr(sample, "edge_attr") and sample.edge_attr is not None:
            ew = sample.edge_attr.numpy().flatten()
            if len(ew) == ei.shape[1]:
                # Normalizza spessore: [0.3, 2.0]
                ew_min, ew_max = ew.min(), ew.max()
                if ew_max > ew_min:
                    edge_weight = 0.3 + 1.7 * (ew - ew_min) / (ew_max - ew_min)
                else:
                    edge_weight = np.ones(len(ew)) * 0.8

        for i in range(ei.shape[1]):
            src, dst = ei[0, i], ei[1, i]
            lw = float(edge_weight[i]) if edge_weight is not None else 0.6
            ax.plot(
                [xs[src], xs[dst]], [ys[src], ys[dst]],
                color="gray", alpha=0.35, linewidth=lw, zorder=1
            )

        # Disegna nodi
        ax.scatter(xs, ys, c=node_colors, s=28, zorder=5,
                   edgecolors="white", linewidths=0.4)

    # Legenda regioni
    region_legend = [
        mpatches.Patch(color="#4C72B0", label="Frontale (F)"),
        mpatches.Patch(color="#55A868", label="Temporale (T)"),
        mpatches.Patch(color="#DD8452", label="Parietale (P)"),
        mpatches.Patch(color="#C44E52", label="Occipitale (O)"),
        mpatches.Patch(color="#8172B2", label="Centrale (C)"),
        mpatches.Patch(color="#999999", label="Altro"),
    ]
    fig.legend(handles=region_legend, loc="lower center", ncol=6, fontsize=8,
               bbox_to_anchor=(0.5, -0.06))

    plt.tight_layout()
    out_path = figures_dir / "eeg08_graph_comparison_trial0.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvato in: {out_path}")

except FileNotFoundError as e:
    print(f"[SKIP] Dataset non disponibili localmente: {e}")
    print("Questa cella produce output solo sulla VM remota dove i dataset sono pre-calcolati.")
    print("Eseguire il notebook EEG_07b_precompute_graphs.ipynb sulla VM prima di rieseguire questa cella.")
except Exception as e:
    print(f"[ERRORE] {type(e).__name__}: {e}")
    import traceback
    traceback.print_exc()

## Conclusioni

### Riepilogo Risultati EEG_08

| Metodo | Architettura | val_bacc | test_bacc | Tempo (min) |
|--------|-------------|----------|-----------|-------------|
| PCC    | ChebGCN_2L      | 0.2527 | 0.2490 | 2.4  |
| PCC    | ChebGCN_3L      | 0.2607 | 0.2445 | 5.1  |
| PCC    | ChebGCN_Skip    | 0.2538 | 0.2524 | 3.2  |
| PLV    | ChebGCN_2L      | 0.2557 | 0.2437 | 3.4  |
| PLV    | ChebGCN_3L      | 0.2522 | 0.2469 | 3.3  |
| PLV    | ChebGCN_Skip    | 0.2503 | 0.2499 | 3.8  |
| wPLI   | ChebGCN_2L      | 0.2511 | 0.2514 | 2.9  |
| wPLI   | ChebGCN_3L      | 0.2528 | 0.2512 | 3.7  |
| wPLI   | ChebGCN_Skip    | 0.2535 | 0.2515 | 2.7  |
| Learned | ChebGCN_Learned | 0.2528 | 0.2430 | 103.7 |
| Dynamic | ChebGCN_Dynamic | 0.2501 | 0.2485 | 123.5 |

### Conclusioni Principali

1. **Tutti i modelli si attestano a chance level** (val_bacc ≈ 0.25): nessun metodo di costruzione del grafo porta a un apprendimento genuino della distinzione semantica.

2. **Class collapse pervasivo**: `val_acc` elevata (fino a 40%) è un artefatto — i modelli predicono la classe maggioritaria. La `val_bacc` è la metrica corretta da monitorare.

3. **Il metodo del grafo non fa differenza**: PCC, PLV, wPLI, Learned e Dynamic producono performance equivalenti in termini di `val_bacc`. Non c'è guadagno nell'usare grafi più complessi.

4. **Trade-off tempo/accuratezza**: Learned (~104 min) e Dynamic (~124 min) sono **~50× più lenti** di PCC/PLV/wPLI (2-5 min) senza alcun vantaggio. Scartarli per i prossimi esperimenti.

5. **Best baseline**: PCC / ChebGCN_3L con val_bacc=0.2607 è il migliore in assoluto, ma il margine sul chance è minimo e probabilmente non significativo.

### Prossimi Passi

- Passare a **EEGNet / EEG Conformer** su segnale raw (EEG_09): architetture end-to-end progettate specificamente per EEG
- Aggiungere **class weights** per contrastare il class collapse
- Esplorare **domain adaptation** cross-soggetto (la variabilità inter-soggetto, ε²=0.85, è il principale ostacolo)